# Load AdventureWorksDW tables into DataFrames

Connects to the Azure SQL database `AdventureWorksDW` on `onlcdb.database.windows.net` from a Microsoft Fabric (PySpark) notebook and loads:

- `dbo.DimProduct`
- `dbo.DimReseller`
- `dbo.FactResellerSales`

## 1. Connection settings

In [ ]:
server   = "onlcdb.database.windows.net"
database = "AdventureWorksDW"
username = "student"
# Password is not stored in the notebook: prompt for it, or fetch it from Azure Key Vault
# (notebookutils.credentials.getSecret("https://<vault>.vault.azure.net/", "<secret-name>"))
from getpass import getpass
password = getpass(f"Password for {username}@{server}: ")

jdbc_url = (
    f"jdbc:sqlserver://{server}:1433;"
    f"database={database};"
    "encrypt=true;trustServerCertificate=false;"
    "hostNameInCertificate=*.database.windows.net;loginTimeout=30;"
)

connection_properties = {
    "user": username,
    "password": password,
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver",
}

## 2. Helper to read a table into a Spark DataFrame

In [ ]:
def load_table(table_name):
    """Read a SQL table over JDBC into a Spark DataFrame."""
    return spark.read.jdbc(url=jdbc_url, table=table_name, properties=connection_properties)

## 3. Load the tables

In [ ]:
df_product        = load_table("dbo.DimProduct")
df_reseller       = load_table("dbo.DimReseller")
df_reseller_sales = load_table("dbo.FactResellerSales")

for name, df in [
    ("DimProduct", df_product),
    ("DimReseller", df_reseller),
    ("FactResellerSales", df_reseller_sales),
]:
    print(f"{name}: {df.count():,} rows, {len(df.columns)} columns")

## 4. Preview the data

In [ ]:
display(df_product.limit(10))

In [ ]:
display(df_reseller.limit(10))

In [ ]:
display(df_reseller_sales.limit(10))

## 5. (Optional) Convert to pandas DataFrames

Fine for the dimension tables; use `.limit(...)` first on `FactResellerSales` if memory is a concern.

In [ ]:
pdf_product        = df_product.toPandas()
pdf_reseller       = df_reseller.toPandas()
pdf_reseller_sales = df_reseller_sales.toPandas()

pdf_product.head()

## 6. Upsert into Lakehouse Delta tables

Requires a default Lakehouse attached to this notebook. If a target table doesn't exist yet it is created; otherwise rows are merged on the key columns (matching rows are updated, new rows are inserted).

In [ ]:
from delta.tables import DeltaTable

def upsert_to_lakehouse(df, table_name, key_columns):
    """Merge df into the Lakehouse Delta table on key_columns (create it if missing)."""
    if not spark.catalog.tableExists(table_name):
        df.write.format("delta").saveAsTable(table_name)
        print(f"{table_name}: created with {df.count():,} rows")
        return

    condition = " AND ".join(f"t.`{c}` = s.`{c}`" for c in key_columns)
    (
        DeltaTable.forName(spark, table_name).alias("t")
        .merge(df.alias("s"), condition)
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
    print(f"{table_name}: upserted {df.count():,} source rows")

upsert_to_lakehouse(df_product,        "DimProduct",        ["ProductKey"])
upsert_to_lakehouse(df_reseller,       "DimReseller",       ["ResellerKey"])
upsert_to_lakehouse(df_reseller_sales, "FactResellerSales", ["SalesOrderNumber", "SalesOrderLineNumber"])